[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/gravitational_lensing/gravitational_lensing.ipynb)

# One star, several images

Two masses bend light from a source behind them. As the source moves, its images stretch into arcs, a pair can appear or merge, and the total light rises and falls. Each mass's pull is the inverse of a vector, and the local map of the lens, which carries a small displacement on the sky to the displacement it makes at the source, is a sum of sandwiches: it tells us where images appear and how bright they are.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext
from examples.animation import save_animation
from examples.relativity.gravitational_lensing import render

np.set_printoptions(precision=4, suppress=True)

# The plane of the sky: two Euclidean directions, x and y, in units of the Einstein angle of the total mass.
ga = Algebra("x+y+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Area = ga.gatype.bivector()
LocalMap = ga.gatype((Vector, Vector))                                         # Vector <- Vector


def sky(half_width: float, resolution: int) -> Vector:
    """Directions at the centres of a square of pixels about the optical axis; the centres miss
    the masses, where the deflection is undefined."""
    angles = (np.arange(resolution) + 0.5) * (2 * half_width / resolution) - half_width
    return mv.x * angles[None, :] + mv.y * angles[:, None]                    # [rows, columns] Vector

## 1. Follow each sightline back to the source

Each mass pulls a sightline towards itself by the inverse of the separation, weighted by its mass: `separation.inverse()` points along the separation with the inverse of its length, so the pull weakens with distance. Subtracting the pulls from an observed direction gives the source direction it reaches.

In vector notation the lens equation reads as $\boldsymbol\beta(\boldsymbol\theta)=\boldsymbol\theta-\sum_i m_i(\boldsymbol\theta-\boldsymbol\ell_i)/|\boldsymbol\theta-\boldsymbol\ell_i|^2$.

In [ ]:
# Two equal masses on the x axis, 1.1 Einstein angles apart. The Einstein angle is the unit: a single
# mass directly in front of a source shows it as a ring of that radius.
positions = mv.x * np.array([-0.55, 0.55])                                    # [masses] Vector
masses = np.array([0.5, 0.5])


def deflected(observed: Vector) -> Vector:
    """The source direction each observed direction reaches."""
    # How far the sightline passes from each mass, and on which side.
    separation = observed[..., None] - positions                               # [..., masses] Vector
    # A mass bends light passing it towards itself, by an angle proportional to the mass and inverse
    # to the distance of closest approach. `separation.inverse()` is exactly such a vector: along the
    # separation, of length one over the distance. The light seen from the observed direction left
    # the source from the direction with those bends undone.
    return observed - (masses * separation.inverse()).sum(axis=-1)             # [...] Vector


directions = sky(1.9, 480)                                                    # [rows, columns] Vector
reached = deflected(directions)                                               # [rows, columns] Vector

## 2. The local map, and where an area collapses

Moving a sightline by a small displacement changes each separation by that displacement, and the change of a vector's inverse is a sandwich: `(separation + small).inverse() - separation.inverse()` is, to first order, `-(separation.inverse() >> small)`. So the local map is the identity, which is the open slot `Vector` itself, plus, for each mass, its reflection in the line of the separation, scaled by the inverse square distance. Its outermorphism carries areas; the area ratio is zero on the critical curve and negative where orientation reverses. The lens carries the critical curve to the caustic at the source.

A reflection has no trace, so the local map's trace is two everywhere. A map of the plane with trace two is undone by `(2 * Vector - local) / area`, and its two stretches are one plus and one minus the square root of `1 - area`: the critical curve is where one stretch vanishes.

In matrix notation the local map reads as the Jacobian $A=\partial\boldsymbol\beta/\partial\boldsymbol\theta$ and its area ratio as $\det A$; in the complex notation of lensing theory the sum of the reflections reads as the shear $\gamma=\sum_i m_i/(\bar z-\bar z_i)^2$ and the area ratio as $1-|\gamma|^2$.

In [ ]:
def local_map(observed: Vector) -> LocalMap:
    """The map from a small displacement of each observed direction to the displacement it makes at
    the source: the identity plus each mass's reflection in the line of its separation."""
    separation = observed[..., None] - positions                               # [..., masses] Vector
    return Vector + (masses * (separation.inverse() >> Vector)).sum(axis=-1)   # [...] Vector <- Vector


local = local_map(directions)                                                 # [rows, columns] Vector <- Vector
# The outermorphism carries the unit area; dividing by it reads the signed ratio.
area = local.outermorphism(Area)(mv.xy) / mv.xy                               # [rows, columns] Scalar

In [ ]:
print("trace, smallest and largest:", local.trace().to_array().min(), local.trace().to_array().max())

## 3. The light the observer sees

Each sky pixel shows the source's brightness at the direction its sightline reaches: surface brightness is conserved along a ray. Lensing increases the total light by making images occupy more sky area, and a Gaussian source of finite width keeps that finite at the caustic.

In [ ]:
def brightness(at: Vector, centre: Vector, width: float) -> Scalar:
    """A round Gaussian source of unit peak brightness and the given angular standard deviation."""
    offset = at - centre                                                       # [...] Vector
    return (-(offset | offset) / (2 * width**2)).exp()                         # [...] Scalar


def light(centre: Vector) -> tuple[Scalar, Scalar]:
    """The source about a centre unlensed, and the light each sky pixel shows: the source's
    brightness where its sightline arrives."""
    return brightness(directions, centre, 0.035), brightness(reached, centre, 0.035)


path = mv.x * np.linspace(-0.6, 0.6, 65) + mv.y * 0.12                        # [steps] Vector
# The critical curve is the zero level of the area ratio; the lens carries it to the caustic.
render.draw(directions, area, deflected, positions, light(path[21]));

## 4. Watch a source cross the caustic

As the source crosses the coral caustic, a pair of images appears or disappears on opposite sides of the cyan critical curve. The finite source blends the pair's meeting into an arc. The marked masses are fixed, and their own light is not included.

In [ ]:
movie = save_animation(render.animate(directions, area, deflected, positions, (light(centre) for centre in path)), "gravitational_lensing", 90)
display(Image(filename=str(movie)))

M. Dominik, [*The binary gravitational lens and its extreme cases*](https://arxiv.org/abs/astro-ph/9903014), section 2, gives the mass and angular normalization, lens equation, critical curves and caustics. The sampled window and pixel spacing determine how accurately small images and integrated brightness are resolved.
